In [1]:
import os
os.chdir('/home/wiikai/factor')

import factorlab as lab
import pandas as pd
import numpy as np

filter = lab.Factor("./data/filter-mask", code_level="order_book_id", date_level="date")

In [2]:
# start = '20160101'
# stop = '20240701'
# ptype = 'volume_weighted_price'
# name = '000985.XSHG' 

# price = lab.factor.read(ptype, start=start, stop=stop)
# adjfactor = lab.quotes_day.read("adjfactor", start=start, stop=stop)
# price = price * adjfactor
# pret = price / price.shift(1) - 1
# fret = price.shift(-1) / price -1
# st = lab.quotes_day.read("st", start=start, stop=stop)
# suspended = lab.quotes_day.read("suspended", start=start, stop=stop)
# benchmark = lab.index_weights.read(name ,start=start, stop=stop).isna()
# nonrealizable = st | suspended | (fret.abs() >= 0.1) | (pret.abs() >= 0.1) | benchmark
# nonrealizable = nonrealizable.astype(bool)

In [4]:
start = '20160101'
stop = '20240701'
ptype = 'volume_weighted_price'
name = '000906.XSHG'

price = lab.factor.read(ptype, start=start, stop=stop)
adjfactor = lab.quotes_day.read("adjfactor", start=start, stop=stop)
price = price * adjfactor
pret = price / price.shift(1) - 1
fret = price.shift(-1) / price -1
st = lab.quotes_day.read("st", start=start, stop=stop)
suspended = lab.quotes_day.read("suspended", start=start, stop=stop)
df_500 = lab.index_weights.read('000905.XSHG' ,start=start, stop=stop).isna()
df_300 = lab.index_weights.read('000300.XSHG' ,start=start, stop=stop).isna()
benchmark = df_500 & df_300
nonrealizable = st | suspended | (fret.abs() >= 0.1) | (pret.abs() >= 0.1) | benchmark
nonrealizable = nonrealizable.astype(bool)

In [5]:
nonrealizable.index.name = 'date'
nonrealizable = nonrealizable.sort_index(ascending=False).unstack()
nonrealizable.name = name

if name not in filter.columns:
    filter.add({name: nonrealizable.dtype})
filter.update(nonrealizable)

In [6]:
filter.read()

000985.XSHG  000906.XSHG
order_book_id date                                
000001.XSHE   2016-01-04        False        False
              2016-01-05        False        False
              2016-01-06        False        False
              2016-01-07        False        False
              2016-01-08        False        False
...                               ...          ...
900947.XSHG   2024-07-01         True         True
900948.XSHG   2024-07-01         True         True
900952.XSHG   2024-07-01         True         True
900953.XSHG   2024-07-01         True         True
900957.XSHG   2024-07-01         True         True

[11107192 rows x 2 columns]